In [23]:
# sqlite3: 파일 기반 데이터베이스(SQLite)를 사용할 수 있게 해주는 표준 라이브러리입니다.
# 여기서는 그래프의 상태(메모리)를 저장하기 위해 사용합니다.
import sqlite3

# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# (필요시) OpenAI 채팅 모델을 초기화할 때 쓰는 함수입니다.
# 지금은 Vertex AI 를 사용하므로 직접 사용되지는 않습니다.
from langchain.chat_models import init_chat_model

# Google Cloud Vertex AI 의 Gemini 모델을 사용하기 위한 LangChain 래퍼입니다.
from langchain_google_vertexai import ChatVertexAI

# 대화 메시지(history)를 상태로 다룰 수 있게 해주는 기본 상태 타입입니다.
from langgraph.graph.message import MessagesState

# SqliteSaver: LangGraph의 상태(체크포인트)를
# SQLite 데이터베이스에 저장/복원해 주는 도구입니다.
from langgraph.checkpoint.sqlite import SqliteSaver

# Vertex AI 의 Gemini 모델을 LLM 으로 사용합니다.
# - model_name: 사용할 Gemini 모델 이름
# - project: GCP 프로젝트 ID
# - location: 리전 (예: us-central1)
# - max_tokens: 한 번에 생성할 최대 토큰 수
llm = ChatVertexAI(
  model_name="gemini-2.5-flash-lite",
  project="ai-prompt-evaluator-489612",
  location="us-central1",
  max_tokens=500
)

# SQLite 데이터베이스 파일(memory-sync.db)에 연결합니다.
# - 이 파일 안에 LangGraph 의 상태(체크포인트)가 저장됩니다.
conn = sqlite3.connect(
  "memory-sync.db",
  # check_same_thread=False 옵션은
  # 여러 개의 파이썬 스레드(쓰레드)가 동시에 같은 데이터베이스 연결을 사용할 수 있도록 허용하는 설정입니다.
  # 기본값은 True이며, True일 경우 동일한 스레드에서만 연결 사용이 가능합니다.
  # 여기서 False로 설정한 이유는, LangGraph 또는 여러 작업이 멀티스레드 환경에서
  # 같은 DB 연결을 사용하게 할 때 발생하는 'ProgrammingError'를 피하기 위해서입니다.
  check_same_thread=False,
)

# LangGraph 실행 시 사용할 기본 config 입니다.
# - configurable.thread_id: "0_x" 라는 ID 로 이 대화/세션을 구분합니다.
config =  {
  "configurable": {
    "thread_id": "0_x",
  },
}

In [24]:
# 이 그래프에서 사용할 상태(state)의 모양을 정의합니다.
# MessagesState 를 그대로 상속만 하면, 기본적으로 "messages" 키에
# 대화 내역이 저장됩니다. (추가 필드는 여기서는 사용하지 않습니다.)
class State(MessagesState):
  pass

# 위에서 정의한 State 타입을 사용하는 상태 그래프를 만듭니다.
# 앞으로 노드들을 이 graph_builder 에 등록하고, 간선(실행 순서)을 정의합니다.
graph_builder = StateGraph(State)

In [25]:
# chatbot 노드는 현재까지의 대화 메시지(state["messages"])를
# LLM 에게 보내고, 응답을 다시 messages 에 추가하는 간단한 노드입니다.
# - llm.invoke(...): LLM 에게 대화 내역을 보내서 답변을 생성합니다.
# - 반환: 새로 생성된 응답 메시지를 리스트에 담아 돌려줍니다.
#   (LangGraph 가 기존 messages 와 합쳐서 전체 대화 내역을 관리합니다.)
def chatbot(state: State):
  response = llm.invoke(state["messages"])
  return {
    "messages": [response],
  }

In [26]:
# 그래프에 사용할 노드를 등록합니다.
# - "chatbot": LLM 에게 질문을 보내고 응답을 받는 노드
graph_builder.add_node("chatbot", chatbot)

# 실행 흐름 정의
# START -> chatbot -> END
# 한 번만 챗봇을 호출하고 바로 끝나는 매우 단순한 구조입니다.
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# 그래프를 컴파일하면서 SqliteSaver(checkpointer)를 연결합니다.
# - 이 설정 덕분에 그래프의 상태가 memory-sync.db 에 저장됩니다.
# - 나중에 특정 시점의 상태를 골라서 "시간 여행(time travel)" 하듯이
#   그 지점으로 돌아가거나, 거기서부터 다시 실행할 수 있습니다.
graph = graph_builder.compile(
  checkpointer=SqliteSaver(conn),
)

In [27]:
# 그래프를 한 번 실행해 봅니다.
# - 초기 상태로 messages 에 사용자의 정보를 넣어 줍니다.
#   ("나는 지금 유럽에 살고 있고, 사는 도시는 Valencia 다"라는 내용)
# - config 에 thread_id="0_x" 를 넘겨, 이 대화를 특정 세션으로 구분합니다.
result = graph.invoke(
  {
    "messages": [
      {
        "role": "user",
        "content": "I live in Europe now. And the city I live in is Valecnia.",
      },
    ]
  },
  config=config,
)

In [28]:
# 그래프 실행 결과로 받은 messages 를 예쁘게 출력해 봅니다.
# - user 의 첫 메시지와, LLM 의 첫 응답이 포함되어 있을 수 있습니다.
for message in result["messages"]:
  message.pretty_print()

================================ Human Message =================================

I live in Europe now. And the city I live in is Valecnia.
================================== Ai Message ==================================

That's wonderful! **Valencia** is a fantastic city in Europe. It's known for its beautiful architecture, vibrant culture, delicious food (hello, paella!), and sunny Mediterranean climate.

How long have you been living in Valencia? What do you enjoy most about living there? I'd love to hear more about your experience!
================================ Human Message =================================

I live in Europe now. And the city I live in is Valecnia.
================================== Ai Message ==================================

Ah, thank you for clarifying! It seems there might be a slight typo in the city name. I'm guessing you meant **Valencia**, the beautiful city in Spain.

If you are indeed living in **Valencia, Spain**, then that's a fantastic place to b

In [29]:
# 지금까지 thread_id="0_x" 에 대해 저장된 모든 상태(체크포인트)를 가져옵니다.
# - state_history 는 특정 시점들의 스냅샷들이 순서대로 들어 있는 이터레이터입니다.
# - 이 정보를 이용해서 "어느 시점으로 되돌아갈지"를 고를 수 있습니다.
state_history = graph.get_state_history(config)

In [30]:
# state_history 안에 들어 있는 각 상태 스냅샷을 하나씩 살펴봅니다.
# - state_snapshot.next: 그 시점에서 "다음에 실행될 노드" 정보
# - state_snapshot.values["messages"]: 그 시점에 저장되어 있던 대화 메시지들
# 이 정보를 보고, 나중에 어떤 체크포인트로 "시간 여행"할지 결정할 수 있습니다.
for state_snapshot in list(state_history):
  print(state_snapshot.next)
  print(state_snapshot.values["messages"])
  print("===============\n")

()
[HumanMessage(content='I live in Europe now. And the city I live in is Valecnia.', additional_kwargs={}, response_metadata={}, id='844c5412-7a94-46e1-9d4b-e25203ae914b'), AIMessage(content="That's wonderful! **Valencia** is a fantastic city in Europe. It's known for its beautiful architecture, vibrant culture, delicious food (hello, paella!), and sunny Mediterranean climate.\n\nHow long have you been living in Valencia? What do you enjoy most about living there? I'd love to hear more about your experience!", additional_kwargs={}, response_metadata={'is_blocked': False, 'safety_ratings': [], 'usage_metadata': {'prompt_token_count': 17, 'candidates_token_count': 70, 'total_token_count': 87, 'prompt_tokens_details': [{'modality': 1, 'token_count': 17}], 'candidates_tokens_details': [{'modality': 1, 'token_count': 70}], 'thoughts_token_count': 0, 'cached_content_token_count': 0, 'cache_tokens_details': []}, 'finish_reason': 'STOP', 'avg_logprobs': -0.2797464643205915, 'model_name': 'gem

In [31]:
from langchain_core.messages import HumanMessage

# (예제 코드 일부에서는 to_fork 변수가 별도로 정의되어 있어야 합니다.
#  여기서는 "특정 시점의 설정(to_fork.config)을 기준으로 상태를 수정한다"는 예시로 볼 수 있습니다.)
#
# graph.update_state(...): 이미 저장된 특정 상태(체크포인트)의 내용을 수정하거나,
# 그 상태를 기반으로 새로운 가지(branch)를 만들고 싶을 때 사용합니다.
# 아래 코드는 "나는 Zagreb 에 산다"라는 새로운 메시지로 상태를 덮어쓰는 예시입니다.


# 그래프의 현재 상태(체크포인트)를 가져오는 코드입니다.
# - graph.get_state(config)는, 지금까지 대화와 설정이 저장된 최신 상태를 반환합니다.
# - 이 상태를 'to_fork'라는 변수에 저장해 둡니다.
# → 나중에 이 상태를 바탕으로 새로운 분기(가지)를 만들거나,
#   기존 상태를 수정해서 '시간 여행'을 할 때 사용할 수 있습니다.
to_fork = graph.get_state(config)

graph.update_state(
  to_fork.config,  # pyright: ignore[reportUndefinedVariable]
  {
    "messages": [
      HumanMessage(
        content="I live in Europe now. And the city I live is Zagreb.",
        id="25169a3d-cc86-4a5f-9abd-03d575089a9f",
      )
    ]
  },
)

{'configurable': {'thread_id': '0_x',
  'checkpoint_ns': '',
  'checkpoint_id': '1f120769-10a1-6e5c-8009-f44f4ac67148'}}

In [32]:
# 위에서 update_state 로 수정/분기한 상태에 대해,
# 그 이후의 상태 히스토리를 가져오는 예시입니다.
# - thread_id: "11" 번 세션으로 새 가지(branch)를 관리할 수 있습니다.
# - checkpoint_id: 어떤 체크포인트에서부터 히스토리를 보고 싶은지 지정합니다.
forked_state = graph.get_state_history(
  {
    "configurable": {
      "thread_id": "11",
      "checkpoint_ns": "",
      "checkpoint_id": "1f120766-4229-6523-8005-2e692793b232",
    }
  }
)

# 실제로 forked_state 안에 어떤 상태들이 들어있는지 리스트로 확인해 봅니다.
list(forked_state)

[]

In [33]:
# 특정 체크포인트에서부터 다시 그래프를 실행(재시작)하는 예시입니다.
# - 첫 번째 인자(None): 새 입력이 없더라도, 저장된 상태에서 이어서 실행하겠다는 의미입니다.
# - config.configurable.checkpoint_id: 어떤 시점(체크포인트)에서부터
#   다시 실행을 시작할지 지정합니다.
#   → 마치 "그때 그 시점으로 시간 여행한 뒤, 거기서부터 다시 대화를 이어가기" 같은 효과입니다.
result = graph.invoke(
  None,
  {
    "configurable": {
      "thread_id": "0_x",
      "checkpoint_ns": "",
      "checkpoint_id": "1f120766-4229-6523-8005-2e692793b232",
    }
  },
)

# 재실행 결과로 받은 messages 를 예쁘게 출력해 봅니다.
for message in result["messages"]:
  message.pretty_print()

================================ Human Message =================================

I live in Europe now. And the city I live in is Valecnia.
================================== Ai Message ==================================

That's wonderful! **Valencia** is a fantastic city in Europe. It's known for its beautiful architecture, vibrant culture, delicious food (hello, paella!), and sunny Mediterranean climate.

How long have you been living in Valencia? What do you enjoy most about living there? I'd love to hear more about your experience!
================================ Human Message =================================

I live in Europe now. And the city I live in is Valecnia.
================================== Ai Message ==================================

Ah, thank you for clarifying! It seems there might be a slight typo in the city name. I'm guessing you meant **Valencia**, the beautiful city in Spain.

If you are indeed living in **Valencia, Spain**, then that's a fantastic place to b